In [16]:
# !pip install torchvison

In [17]:
import torch 
import os
from torch.utils.data import DataLoader , Dataset
from torchvision import transforms
from PIL import Image

In [18]:
# imsge Load => trsnsform => dataset of all imgs

class ImageProcessor:
    def __init__(self,root_dir_path , transformations=None):
        self.root_dir_path = root_dir_path
        self.transformations = transformations


        # list of path for all images
        self.all_img_paths = [os.path.join(root_dir_path,img) for img in os.listdir(root_dir_path)]

    def __len__(self):
        return len(self.all_img_paths)

    def __getitem__(self , idx):
        img_path = self.all_img_paths[idx]
        img = Image.open(img_path).convert("RGB")

        if self.transformations:
            img = self.transformations(img)

        return img

In [19]:
root_dir_path = "./img_align_celeba"

transformations = transforms.Compose([
    transforms.CenterCrop(178),  #178*218 ==> 178*178
    transforms.Resize(64),#  178*178 => 64*64
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))  # [-1, 1]
])

In [20]:
dataset = ImageProcessor(root_dir_path, transformations)
print(f"loaded {len(dataset)} images")

loaded 202599 images


In [21]:
dataloader = DataLoader(dataset, batch_size=128, shuffle=True)

## Generator Network

In [22]:
import torch.nn as nn
import torch.optim as optim
import numpy as np

In [23]:
class Generator(nn.Module):
    def __init__(self,z_dim=100,img_channels=3):# 3for (R,G,B) , and 100 means noise vector length
        super(Generator ,self).__init__()
    
        # Fully connected (dense) Layers
        self.model = nn.Sequential(
            nn.Linear(z_dim,256), #100 ==> 256
            nn.ReLU(),
    
            nn.Linear(256,512),
            nn.ReLU(),
    
            nn.Linear(512,1024),
            nn.ReLU(),
    
            nn.Linear(1024,64*64*img_channels),
            nn.Tanh() # becouse it give value in range of [-1,1]
            
        )

    def forward(self,z):
        img =self.model(z)
        img = img.view(img.size(0),3,64,64)
        return img

         # fake img => 64 x 64 x 3 x batch_size

## Discriminator Network

In [24]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=3): # 3 is for RGB
        super(Discriminator, self).__init__()

        # fully connected (dense) layers
        self.model = nn.Sequential(
            nn.Flatten(), # 4D tensor => 1D
            
            nn.Linear(img_channels * 64 * 64, 1024),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(1024, 512), 
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(512, 256), 
            nn.LeakyReLU(0.2, inplace=True),

            nn.Linear(256, 1), 
            nn.Sigmoid() # probability of being real/fake 
        )

    def forward(self, img):
        return self.model(img)

In [25]:
GAN_loss = nn.BCELoss()

generator = Generator()
g_optimizer = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))

discriminator = Discriminator()
d_optimizer = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

In [26]:
import torch

# Device selection order: MPS → CUDA → Intel XPU → AMD HIP → CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")        # Apple Silicon (Mac)
elif torch.cuda.is_available():
    device = torch.device("cuda")       # NVIDIA GPU
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")        # Intel GPU (XPU)
elif hasattr(torch.backends, "hip") and torch.backends.hip.is_available():
    device = torch.device("hip")        # AMD GPU (HIP)
else:
    device = torch.device("cpu")        # CPU fallback

print(f"device is {device}")


device is xpu


In [27]:
generator = generator.to(device)
discriminator = discriminator.to(device)

## Training the GAN

In [28]:
def train(generator, discriminator, dataloader, epochs=10):

    for epoch in range(epochs):
        for i, imgs in enumerate(dataloader):
            real_imgs = imgs.to(device)
            batch_size = real_imgs.size(0)

            # creates real imgs labels & fake imgs labels
            real_labels = torch.ones(batch_size, 1).to(device) # [1, 1, 1....]
            fake_labels = torch.zeros(batch_size, 1).to(device) # [0, 0, 0....]

            # Train the Discriminator
            d_optimizer.zero_grad()

            fake_imgs = generator(torch.randn(batch_size, 100).to(device))

            real_loss = GAN_loss(discriminator(real_imgs), real_labels)
            fake_loss = GAN_loss(discriminator(fake_imgs.detach()), fake_labels)

            d_loss = (real_loss + fake_loss) / 2

            d_loss.backward()
            d_optimizer.step()

            # Train the Generator
            g_optimizer.zero_grad()

            g_loss = GAN_loss(discriminator(fake_imgs), real_labels)

            g_loss.backward()
            g_optimizer.step()

            if i % 50 == 0:
                print(f"for epoch: {epoch+1}/{epochs}... batch: {i+1}... G-loss:{g_loss}.... D-loss: {d_loss}")

        # save generated imgs for each epoch
        save_generated_images(generator, epoch, device) 

In [29]:
import matplotlib.pyplot as plt
import torchvision

def save_generated_images(generator, epoch, device, num_imgs=8):
    z = torch.randn(num_imgs, 100).to(device)
    generated_imgs = generator(z).detach().cpu()

    grid = torchvision.utils.make_grid(generated_imgs, nrow=4, normalize=True)

    plt.imshow(np.transpose(grid, (1, 2, 0)))
    plt.title(f"epoch {epoch+1}")
    plt.axis("off")
    plt.show()

In [ ]:
train(generator, discriminator, dataloader, epochs=5)

for epoch: 1/5... batch: 1... G-loss:0.6827506422996521.... D-loss: 0.6986393332481384
for epoch: 1/5... batch: 51... G-loss:1.0978825092315674.... D-loss: 0.3014299273490906
